# Files

The [`std` and `os` modules](https://bellard.org/quickjs/quickjs.html#Standard-library) of QuickJS read and
write files. Their API is the one of QuickJS, close to the C library, and not the `fs` module of Node.js.

The relative paths start from the directory of the notebook. In JupyterLite, the files are the ones of the
file browser, stored in the browser: what the kernel writes shows up in the file browser. This needs the
service worker of JupyterLite, which browsers only start when the site is served over HTTPS or from `localhost`.

In [ ]:
import * as std from 'std';
import * as os from 'os';

os.getcwd()

Most functions of `os` return their result with an error number, `0` when there is no error.

## Text files

`std.open` opens a file like `fopen` in C, and returns a `FILE` object. `std.loadFile` reads a whole file:

In [ ]:
const notes = std.open('notes.txt', 'w');
notes.puts('QuickJS writes files.\n');
notes.printf('%d + %d = %d\n', 2, 3, 2 + 3);
notes.close();

std.loadFile('notes.txt')

The `a` mode appends to a file, and `getline` reads a file line by line:

In [ ]:
const more = std.open('notes.txt', 'a');
more.puts('One more line.\n');
more.close();

const reader = std.open('notes.txt', 'r');
const lines = [];
for (let line = reader.getline(); line !== null; line = reader.getline()) {
  lines.push(line);
}
reader.close();
lines

## Errors

The functions do not throw errors. `std.open` and `std.loadFile` return `null` when a file cannot be read,
and `std.open` gives the error number in its third argument:

In [ ]:
const error = {};
const missing = std.open('missing.txt', 'r', error);
[missing, error.errno === std.Error.ENOENT, std.strerror(error.errno)]

The functions of `os` return `0` when they succeed, and a negative error number otherwise. Creating the
same directory twice fails with `EEXIST`:

In [ ]:
[os.mkdir('reports'), os.mkdir('reports') === -std.Error.EEXIST]

## JSON and CSV files

`modules/planets.json` comes with the example notebooks:

In [ ]:
const planets = JSON.parse(std.loadFile('modules/planets.json'));

const csv = std.open('reports/planets.csv', 'w');
csv.puts('name,radius,moons\n');
for (const { name, radius, moons } of planets) {
  csv.puts(`${name},${radius},${moons}\n`);
}
csv.close();

std.loadFile('reports/planets.csv')

Reading the CSV file back:

In [ ]:
const [header, ...rows] = std.loadFile('reports/planets.csv').trim().split('\n');
const columns = header.split(',');
console.table(rows.map(row => Object.fromEntries(row.split(',').map((value, i) => [columns[i], value]))));

In [ ]:
const largest = planets.reduce((a, b) => (b.radius > a.radius ? b : a));
const summary = std.open('reports/summary.json', 'w');
summary.puts(JSON.stringify({ planets: planets.length, largest: largest.name }, null, 2));
summary.close();

JSON.parse(std.loadFile('reports/summary.json'))

## Binary files

`write` and `read` take an `ArrayBuffer`, a position in it and a length. `seek` and `tell` move in the file:

In [ ]:
const bytes = new Uint8Array(256).map((_, i) => i);
const output = std.open('reports/bytes.bin', 'wb');
output.write(bytes.buffer, 0, bytes.length);
output.close();

const input = std.open('reports/bytes.bin', 'rb');
input.seek(0, std.SEEK_END);
const size = input.tell();
input.seek(-4, std.SEEK_END);
const last = new Uint8Array(4);
input.read(last.buffer, 0, last.length);
input.close();

[size, last]

## Images

`display.image` shows an image file, here an SVG chart written by the cell:

In [ ]:
const bars = planets.map(({ name, radius }, i) => {
  const height = (radius / largest.radius) * 100;
  return `<rect x="${20 + i * 60}" y="${110 - height}" width="40" height="${height}" fill="steelblue"/>` +
    `<text x="${40 + i * 60}" y="128" font-size="11" text-anchor="middle">${name}</text>`;
});
const chart = std.open('reports/radius.svg', 'w');
chart.puts(`<svg xmlns="http://www.w3.org/2000/svg" width="260" height="136">${bars.join('')}</svg>`);
chart.close();

display.image('reports/radius.svg');

## Directories

`os.readdir` lists a directory, and `os.stat` gives the size, the dates and the type of a file:

In [ ]:
const isDirectory = path => (os.stat(path)[0].mode & os.S_IFMT) === os.S_IFDIR;

const [names] = os.readdir('reports');
console.table(names.filter(name => !name.startsWith('.')).sort().map(name => {
  const [info] = os.stat(`reports/${name}`);
  return { name, size: info.size, modified: new Date(info.mtime).toISOString() };
}));

A small function lists all the files under a directory:

In [ ]:
function walk(dir) {
  return os.readdir(dir)[0]
    .filter(name => !name.startsWith('.'))
    .sort()
    .flatMap(name => {
      const path = `${dir}/${name}`;
      return isDirectory(path) ? walk(path) : [path];
    });
}

walk('reports')

## Moving and removing files

`os.rename` moves a file, and `os.remove` removes a file or an empty directory:

In [ ]:
os.rename('notes.txt', 'reports/notes.txt');

for (const path of walk('reports')) {
  os.remove(path);
}
os.remove('reports');

[std.loadFile('notes.txt'), os.readdir('.')[0].includes('reports')]